In [2]:
import pandas as pd
import os

In [9]:
import urllib.request

url = "https://raw.githubusercontent.com/gustavsganzerla/cdbprom/main/training_data.csv"
urllib.request.urlretrieve(url, "training_data.csv")


('training_data.csv', <http.client.HTTPMessage at 0x1424f7820>)

In [13]:
df_train = pd.read_csv("./Datasets/training_data.csv")
print(f"✅ File loaded successfully. Shape: {df_train.shape}")
df_train.head()

✅ File loaded successfully. Shape: (31259, 101)


,Unnamed: 0,0,1,2,3,4,5,6,7,8,...,90,91,92,93,94,95,96,97,98,label
0,0,-1.44,-1.84,-2.27,-1.84,-1.45,-1.30,-1.84,-1.30,-1.45,...,-1.30,-1.00,-1.00,-1.30,-1.84,-1.84,-2.27,-1.28,-1.00,1
1,1,-1.45,-1.45,-0.88,-1.00,-1.44,-2.27,-1.45,-1.45,-1.84,...,-1.28,-1.00,-1.44,-1.84,-2.27,-1.45,-1.30,-1.44,-0.58,1
2,2,-1.84,-1.45,-0.88,-1.44,-1.84,-1.44,-1.00,-1.44,-2.27,...,-0.88,-1.00,-1.44,-2.27,-1.84,-1.28,-1.00,-1.44,-1.84,1
3,3,-2.24,-1.30,-0.88,-1.28,-2.24,-2.27,-1.84,-1.28,-1.44,...,-1.84,-1.28,-1.28,-1.28,-1.44,-1.84,-1.84,-1.84,-1.30,1
4,4,-1.45,-1.45,-0.88,-0.58,-1.30,-2.27,-1.84,-1.45,-0.88,...,-1.28,-0.58,-1.30,-1.44,-1.28,-1.28,-1.44,-1.44,-1.44,1


In [7]:

# ---------- 1. Verify CDB_prom.csv (Training dataset from CDBProm) ----------
print("="*50)
print("Verifying CDB_prom.csv (Training dataset)")
print("="*50)

try:
    df_train = pd.read_csv("./Datasets/training_data.csv")
    print(f"✅ File loaded successfully. Shape: {df_train.shape}")
    
    # Expected columns: sequence, label, etc. Check based on CDBProm format
    print(f"Columns: {list(df_train.columns)}")
    
    # Count promoters (label=1) and non-promoters (label=0)
    if 'label' in df_train.columns:
        n_pos = (df_train['label'] == 1).sum()
        n_neg = (df_train['label'] == 0).sum()
        print(f"✅ Promoters (label=1): {n_pos}")
        print(f"✅ Non-promoters (label=0): {n_neg}")
        if n_pos == 15654 and n_neg == 15654:
            print("✅ Correct: 15,654 promoters and 15,654 non-promoters")
        else:
            print(f"⚠️ Warning: Expected 15,654 each, got {n_pos} / {n_neg}")
    else:
        print("⚠️ Warning: 'label' column not found. Check column names.")
    
    # Check sequence lengths (should be 60 bp)
    if 'sequence' in df_train.columns:
        seq_len = df_train['sequence'].str.len().unique()
        print(f"✅ Sequence lengths: {seq_len}")
        if len(seq_len) == 1 and seq_len[0] == 60:
            print("✅ All sequences are 60 bp")
        else:
            print("⚠️ Warning: Sequences not all 60 bp")
    else:
        print("⚠️ Warning: 'sequence' column not found.")
        
except FileNotFoundError:
    print("❌ File not found: CDB_prom.csv")
except Exception as e:
    print(f"❌ Error loading file: {e}")

Verifying CDB_prom.csv (Training dataset)
✅ File loaded successfully. Shape: (31259, 101)
Columns: ['Unnamed: 0', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', 'label']
✅ Promoters (label=1): 15654
✅ Non-promoters (label=0): 15605
⚠️ Warning: Expected 15,654 each, got 15654 / 15605
⚠️ Warning: 'sequence' column not found.


In [12]:
import pandas as pd

df_regulon = pd.read_csv("./Datasets/PromoterSet.tsv", sep='\t', encoding='utf-8', on_bad_lines='skip')
print(f"✅ File loaded successfully. Shape: {df_regulon.shape}")

✅ File loaded successfully. Shape: (32, 1)


In [4]:
# ---------- 2. Verify PromoterSet.tsv (RegulonDB validation set) ----------
print("\n" + "="*50)
print("Verifying PromoterSet.tsv (RegulonDB E. coli promoters)")
print("="*50)

try:
    df_regulon = pd.read_csv("./Datasets/PromoterSet.tsv", sep='\t', encoding='utf-8', on_bad_lines='skip')
    print(f"✅ File loaded successfully. Shape: {df_regulon.shape}")
    
    # Expected number: 3,861 promoters (approx)
    print(f"Number of rows: {len(df_regulon)}")
    if 3800 <= len(df_regulon) <= 3900:
        print(f"✅ Approximately correct (expected ~3861)")
    else:
        print(f"⚠️ Warning: Expected ~3861, got {len(df_regulon)}")
    
    # Check if organism is E. coli (if column exists)
    if 'organism' in df_regulon.columns:
        organisms = df_regulon['organism'].unique()
        print(f"Organisms present: {organisms}")
        if 'Escherichia coli' in organisms or 'E. coli' in str(organisms):
            print("✅ Contains E. coli promoters")
    elif 'species' in df_regulon.columns:
        print(f"Species: {df_regulon['species'].unique()}")
    else:
        print("⚠️ No organism/species column found. Check column names.")
    
    print(f"First few column names: {list(df_regulon.columns[:5])}")
    
except FileNotFoundError:
    print("❌ File not found: PromoterSet.tsv")
except Exception as e:
    print(f"❌ Error loading file: {e}")




Verifying PromoterSet.tsv (RegulonDB E. coli promoters)
✅ File loaded successfully. Shape: (32, 1)
Number of rows: 32
⚠️ Warning: Expected ~3861, got 32
⚠️ No organism/species column found. Check column names.
First few column names: ['# License']


In [6]:
# ---------- 3. Verify Data_1 folder (Zhang et al. benchmark) ----------
print("\n" + "="*50)
print("Verifying Data_1 folder (Zhang et al. 2022 benchmark)")
print("="*50)

folder_path = "./Datasets/Data_1"  # Change if folder is named differently
expected_files = ['A.thaliana.xlsx', 'B.subtilis.xlsx', 'D.melanogaster.xlsx', 
                  'E.coli.xlsx', 'H.sapiens.xlsx', 'M.musculus.xlsx']

if os.path.exists(folder_path) and os.path.isdir(folder_path):
    files_found = os.listdir(folder_path)
    print(f"Files found in folder: {files_found}")
    
    for fname in expected_files:
        if fname in files_found:
            print(f"✅ {fname} found")
        else:
            print(f"❌ {fname} missing")
    
    # Load one bacterial file as example (e.g., E.coli.xlsx or B.subtilis.xlsx)
    for species_file in ['E.coli.xlsx', 'B.subtilis.xlsx']:
        if species_file in files_found:
            filepath = os.path.join(folder_path, species_file)
            try:
                df_species = pd.read_excel(filepath)
                print(f"\n✅ Loaded {species_file}: shape {df_species.shape}")
                print(f"Columns: {list(df_species.columns)}")
                # Check for sequence column
                if 'sequence' in df_species.columns or 'seq' in df_species.columns:
                    seq_col = 'sequence' if 'sequence' in df_species.columns else 'seq'
                    lengths = df_species[seq_col].str.len().unique()
                    print(f"Sequence lengths: {lengths}")
                else:
                    print("⚠️ No sequence column found. Check Excel structure.")
            except Exception as e:
                print(f"❌ Error loading {species_file}: {e}")
else:
    print(f"❌ Folder '{folder_path}' not found. Please check the path.")

print("\n" + "="*50)
print("Verification complete.")




Verifying Data_1 folder (Zhang et al. 2022 benchmark)
Files found in folder: ['H.sapiens.xlsx', 'E.coli.xlsx', 'D.melanogaster.xlsx', 'M.musculus.xlsx', 'A.thaliana.xlsx', 'B.subtilis.xlsx']
✅ A.thaliana.xlsx found
✅ B.subtilis.xlsx found
✅ D.melanogaster.xlsx found
✅ E.coli.xlsx found
✅ H.sapiens.xlsx found
✅ M.musculus.xlsx found
❌ Error loading E.coli.xlsx: `Import openpyxl` failed.  Use pip or conda to install the openpyxl package.
❌ Error loading B.subtilis.xlsx: `Import openpyxl` failed.  Use pip or conda to install the openpyxl package.

Verification complete.


In [ ]:
import pandas as pd

# 1. Training data
df_train = pd.read_csv("training_data.csv")
print(f"Training data shape: {df_train.shape}")
print(f"Columns: {list(df_train.columns)}")
print(f"Promoters: {(df_train['label']==1).sum()}, Non-promoters: {(df_train['label']==0).sum()}")
print(f"Sequence length: {df_train['sequence'].str.len().unique()}")

# 2. RegulonDB validation
df_reg = pd.read_csv("PromoterSet.txt", sep='\t')
print(f"\nRegulonDB shape: {df_reg.shape}")
print(f"First few rows:\n{df_reg.head()}")

# 3. Zhang et al. benchmark (after installing openpyxl)
df_ecoli = pd.read_excel("Data_1/E.coli.xlsx")
df_bsub = pd.read_excel("Data_1/B.subtilis.xlsx")
print(f"\nE.coli benchmark shape: {df_ecoli.shape}")
print(f"B.subtilis benchmark shape: {df_bsub.shape}")